# Molecular Combinatorial Shift Keying (MoCSK) Framework

Unified pipeline for molecular communication simulation, modulation, decoding, and benchmarking.

### Implemented schemes
1. **Permutation MoCSK**: full molecule permutations decoded by joint slot assignment.
2. **MoCSK**: ordered subsets, including the empty symbol, decoded with a channel-derived fixed threshold.
3. **E-MoCSK**: sparse slot and molecule patterns decoded with empty-slot assignment.

Moving-average slot scores use the minimum signal in a valley half a slot before the expected peak. The detector is limited when slot spacing is shorter than the pulse width.


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import config
from modulation import (
    run_permutation,
    run_combination_fixed,
    run_sparse,
    get_all_permutations,
    get_all_combinations,
    get_all_sparse_patterns,
)
from channel.diffusion import ensure_experiment_templates
from experiments.sweep import run_requested_sweeps

print("Physical Configuration:")
print(f"  Distance r0           : {config.R0} µm")
print(f"  Receiver radius rr    : {config.RR} µm")
print(f"  Diffusion coef D      : {config.D} µm²/s")
print(f"  Theoretical Peak Time : {config.T_PEAK_THEORY:.4f} s")
print(f"  Template directory    : {config.CUSTOM_DIR}")

## 1. Permutation Scheme
Each '1' bit carries a random full permutation of all $K$ molecule types. Decoded by finding the arrival peak order.

In [ ]:
test_bits = "10110"

res_perm = run_permutation(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)


## 2. MoCSK Ordered-Subset Scheme
Each active symbol carries a uniformly sampled ordered subset, including the empty symbol. The threshold is one half of the isolated-pulse channel peak.


In [ ]:
res_combo_fixed = run_combination_fixed(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)


## 3. E-MoCSK Sparse Pattern Scheme
Slots are partially filled with distinct molecule types. The decoder assigns molecules jointly and uses dummy columns for empty slots.


In [ ]:
res_sparse = run_sparse(
    bit_sequence=test_bits,
    num_molecule_types=5,
    delay_between_symbols=2.5,
    delay_between_molecules=0.4,
    plot=True,
)

## 4. Monte Carlo SER and BER-Approximate Goodput Sweep
Runs pooled Monte Carlo simulations with 95% Wilson confidence intervals for the three implemented schemes. The N_Tx sweep uses Ts=1.0 s for every K. BER-approximate goodput uses a uniform-symbol-error approximation.


In [ ]:
results = run_requested_sweeps(
    schemes=["permutation", "combo_fixed", "sparse"],
    molecule_type_counts=[5],
    n_tx_values=[500, 1000],
    ts_sweep_n_tx=500,
    ts_values=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    seq_length=100,
    num_trials=10,
    n_exp_templates=100,
)
